# Collective Analysis: Forest Plot

This notebook creates forest plots for stop category contrasts based on collective model coefficients.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.ticker import FormatStrFormatter

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 6)

## Load Data

In [ ]:
# Load collective coefficients
coefficients_df = pd.read_csv('../shared/output/collective_coefficients.csv')

print(f"Data shape: {coefficients_df.shape}")
coefficients_df.head(20)

## Prepare Data for Forest Plot

Filter coefficients by p-value < 0.1 (including marginally significant results) and organize by contrast.

In [ ]:
# Calculate p-value from z value if not already present
if 'Pr(>|z|)' in coefficients_df.columns:
    coefficients_df['p_value'] = coefficients_df['Pr(>|z|)']
elif 'z value' in coefficients_df.columns:
    coefficients_df['p_value'] = 2 * (1 - stats.norm.cdf(np.abs(coefficients_df['z value'])))

# Don't filter by p-value yet - we'll include specific predictors regardless of p-value
sig_coef = coefficients_df.copy()

print(f"Total coefficients: {len(sig_coef)}")

# Parse term names to extract contrast and predictor
def parse_term(term):
    if '~' in term:
        parts = term.split('~')
        contrast = parts[0]
        predictor = parts[1]
    else:
        contrast = 'baseline'
        predictor = term
    return contrast, predictor

sig_coef[['contrast', 'predictor']] = sig_coef['term'].apply(lambda x: pd.Series(parse_term(x)))

# Define contrast pairs and their labels
contrast_mapping = {
    'asp': 'Aspirated/Lenis',
    'fortis': 'Fortis/Lenis'
}

# Filter for main contrasts
sig_coef = sig_coef[sig_coef['contrast'].isin(['asp', 'fortis'])]
sig_coef['contrast_label'] = sig_coef['contrast'].map(contrast_mapping)

# Clean predictor names
predictor_mapping = {
    'svot': 'VOT',
    'sf0': 'f0',
    'poalab': 'Labial',
    'poador': 'Dorsal',
    '(Intercept)': 'Intercept',
    'svot:sage': 'Age:VOT',
    'svot:genderMale': 'Male:VOT',
    'sage': 'Age',
    'genderMale': 'Male'
}

# Apply mapping with fallback
sig_coef['predictor_label'] = sig_coef['predictor'].apply(
    lambda x: predictor_mapping.get(x, x)
)

sig_coef

## Create Forest Plot

In [ ]:
# Create separate dataframes for each contrast comparison
asp_data = sig_coef[sig_coef['contrast'] == 'asp'].copy()
fortis_data = sig_coef[sig_coef['contrast'] == 'fortis'].copy()

# For Aspirated/Fortis, we subtract fortis from asp coefficients
# Use original 'predictor' column for merging to avoid duplicates
merged = asp_data.merge(fortis_data, on='predictor', suffixes=('_asp', '_fortis'), how='outer')
asp_fortis_data = []

for _, row in merged.iterrows():
    est_asp = row['Estimate_asp'] if pd.notna(row['Estimate_asp']) else 0
    est_fortis = row['Estimate_fortis'] if pd.notna(row['Estimate_fortis']) else 0
    se_asp = row['Std. Error_asp'] if pd.notna(row['Std. Error_asp']) else 0
    se_fortis = row['Std. Error_fortis'] if pd.notna(row['Std. Error_fortis']) else 0
    
    # Get predictor label (prefer asp version, then fortis)
    pred_label = row['predictor_label_asp'] if pd.notna(row['predictor_label_asp']) else row['predictor_label_fortis']
    
    asp_fortis_data.append({
        'contrast_label': 'Aspirated/Fortis (K)',
        'predictor_label': pred_label,
        'Estimate': est_asp - est_fortis,
        'Std. Error': np.sqrt(se_asp**2 + se_fortis**2)
    })

asp_fortis = pd.DataFrame(asp_fortis_data)

# Update other contrast labels to include (K)
sig_coef_plot = sig_coef[['contrast_label', 'predictor_label', 'Estimate', 'Std. Error']].copy()
sig_coef_plot['contrast_label'] = sig_coef_plot['contrast_label'] + ' (K)'

# Combine all data
all_plot_data = pd.concat([asp_fortis, sig_coef_plot], ignore_index=True)

# Define specific predictors and their order for each contrast
contrast_predictor_order = {
    'Aspirated/Fortis (K)': ['VOT', 'f0', 'Labial', 'Dorsal', 'Age:VOT', 'Male:VOT'],
    'Aspirated/Lenis (K)': ['VOT', 'f0', 'Labial', 'Dorsal', 'Male:VOT'],
    'Fortis/Lenis (K)': ['VOT', 'f0', 'Labial', 'Dorsal', 'Age:VOT', 'Male:VOT']
}

# Get unique contrasts in desired order
unique_contrasts = ['Aspirated/Fortis (K)', 'Aspirated/Lenis (K)', 'Fortis/Lenis (K)']
n_contrasts = len(unique_contrasts)

# Create subplots - height is 2.5x width, no shared y-axis
fig, axes = plt.subplots(1, n_contrasts, figsize=(3*n_contrasts, 6))
if n_contrasts == 1:
    axes = [axes]

for idx, contrast in enumerate(unique_contrasts):
    ax = axes[idx]
    
    # Get data for this contrast
    contrast_data = all_plot_data[all_plot_data['contrast_label'] == contrast].copy()
    
    # Filter to only include specified predictors for this contrast
    predictor_order = contrast_predictor_order[contrast]
    contrast_data = contrast_data[contrast_data['predictor_label'].isin(predictor_order)]
    
    # Sort by the specified order
    contrast_data['predictor_order'] = contrast_data['predictor_label'].apply(
        lambda x: predictor_order.index(x) if x in predictor_order else 999
    )
    contrast_data = contrast_data.sort_values('predictor_order')
    
    # Calculate 95% CI
    contrast_data['ci_lower'] = contrast_data['Estimate'] - 1.96 * contrast_data['Std. Error']
    contrast_data['ci_upper'] = contrast_data['Estimate'] + 1.96 * contrast_data['Std. Error']
    
    # X positions
    x_pos = np.arange(len(contrast_data))
    
    # Plot points and error bars (swapped axes)
    ax.errorbar(x_pos, contrast_data['Estimate'], 
               yerr=[contrast_data['Estimate'] - contrast_data['ci_lower'],
                     contrast_data['ci_upper'] - contrast_data['Estimate']],
               fmt='o', markersize=8, capsize=5, capthick=2,
               color='black', ecolor='black', linewidth=2)
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=2, alpha=0.5)
    
    # Set tick labels for each subplot
    ax.set_xticks(x_pos)
    ax.set_xticklabels(contrast_data['predictor_label'], fontsize=11, rotation=45, ha='right')
    
    # Set title for each subplot
    ax.set_title(contrast, fontsize=13, fontweight='bold')
    
    # Set x-axis limits with more padding
    ax.set_xlim(-0.8, len(contrast_data) - 0.2)
    
    # Grid
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_axisbelow(True)
    
    # Set black border for each subplot
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)

# Add common axis labels for the entire figure
fig.text(0.5, 0, 'Fixed effect', ha='center', fontsize=12, fontweight='bold')
fig.text(0.02, 0.5, 'Coefficient', va='center', rotation='vertical', fontsize=12, fontweight='bold')

# Main title and note
fig.suptitle('Forest Plot: Stop Category Contrasts',
            fontsize=15, fontweight='bold', y=0.98)
fig.text(0.5, -0.08, 
        "Note: 'VOT' and 'f0' are scaled variables. Error bars represent 95% confidence intervals.",
        ha='center', fontsize=10, style='italic')
fig.text(0.5, -0.11,
        "Only main effects and specific VOT interactions are shown. Scales vary by panel for better visibility.",
        ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.subplots_adjust(left=0.08, bottom=0.15)
plt.show()

# Prediction vs Actual Response Probabilities

Load predictions and visualize predicted vs actual response probabilities by VOT and F0 bins.


In [ ]:
# Load predictions
predictions_df = pd.read_csv('../shared/output/collective_predictions.csv')

# Merge with original data to get additional information
original_data = pd.read_csv('../shared/output/perception_preprocessed_data.csv')

# Create F0 bins similar to the image (F0 1-2, F0 3-4, F0 5-6, F0 7-8)
def bin_f0(f0_value):
    if f0_value <= 2:
        return 'F0 1-2'
    elif f0_value <= 4:
        return 'F0 3-4'
    elif f0_value <= 6:
        return 'F0 5-6'
    else:
        return 'F0 7-8'

predictions_df['f0_bin'] = predictions_df['f0'].apply(bin_f0)

# Calculate actual probabilities: for each (vot, f0_bin, response), count occurrences
actual_probs = predictions_df.groupby(['vot', 'f0_bin', 'actual_response']).size().reset_index(name='count')
actual_probs['total'] = actual_probs.groupby(['vot', 'f0_bin'])['count'].transform('sum')
actual_probs['probability'] = actual_probs['count'] / actual_probs['total']
actual_probs = actual_probs.rename(columns={'actual_response': 'response'})

# Calculate predicted probabilities: average the model's predicted probabilities for each (vot, f0_bin, response)
# This gives us smooth curves because we use the actual probability values from the model
predicted_probs_list = []

for response in ['asp', 'fortis', 'lenis']:
    prob_col = f'prob_{response}'
    grouped = predictions_df.groupby(['vot', 'f0_bin'])[prob_col].mean().reset_index()
    grouped['response'] = response
    grouped['probability'] = grouped[prob_col]
    grouped = grouped[['vot', 'f0_bin', 'response', 'probability']]
    predicted_probs_list.append(grouped)

predicted_probs = pd.concat(predicted_probs_list, ignore_index=True)

print("Actual probabilities sample:")
print(actual_probs.head(10))
print("\nPredicted probabilities sample:")
print(predicted_probs.head(10))


In [ ]:
predicted_probs[predicted_probs['f0_bin'] == "F0 7-8"]

In [ ]:
# Define color palette for F0 bins
f0_colors = {
    'F0 1-2': '#1f77b4',  # blue
    'F0 3-4': '#ff7f0e',  # orange
    'F0 5-6': '#2ca02c',  # green
    'F0 7-8': '#d62728'   # red (light version)
}

# Define response full names
response_names = {
    'asp': 'Aspirated',
    'fortis': 'Fortis',
    'lenis': 'Lenis'
}

# Create 1x3 grid with shared y-axis
fig, axes = plt.subplots(1, 3, figsize=(20, 8), sharey=True)
responses = ['asp', 'fortis', 'lenis']

for idx, response in enumerate(responses):
    ax = axes[idx]
    
    # Filter data for this response
    actual_data = actual_probs[actual_probs['response'] == response].copy()
    predicted_data = predicted_probs[predicted_probs['response'] == response].copy()
    
    # Plot scatter for actual data (grouped by f0_bin)
    for f0_bin in ['F0 1-2', 'F0 3-4', 'F0 5-6', 'F0 7-8']:
        data = actual_data[actual_data['f0_bin'] == f0_bin]
        if len(data) > 0:
            ax.scatter(data['vot'], data['probability'], 
                      alpha=0.6, s=100, label=f0_bin,
                      color=f0_colors[f0_bin])
    
    # Plot lines for predicted data (grouped by f0_bin)
    for f0_bin in ['F0 1-2', 'F0 3-4', 'F0 5-6', 'F0 7-8']:
        data = predicted_data[predicted_data['f0_bin'] == f0_bin].sort_values('vot')
        if len(data) > 0:
            ax.plot(data['vot'], data['probability'], 
                   linewidth=3, label=f'{f0_bin} (pred)',
                   color=f0_colors[f0_bin], linestyle='-')
    
    # Set labels and title
    ax.set_xlabel('VOT (ms)', fontsize=12, fontweight='bold')
    if idx == 0:
        ax.set_ylabel('Probability', fontsize=12, fontweight='bold')
    
    ax.set_title(response_names[response], fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

# Add legend
handles = [plt.scatter([], [], alpha=0.6, s=100, color=f0_colors[f0_bin], label=f0_bin)
           for f0_bin in ['F0 1-2', 'F0 3-4', 'F0 5-6', 'F0 7-8']]
handles += [plt.Line2D([0], [0], color='black', linewidth=3, label='Predicted')]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=5, fontsize=11)

fig.suptitle('Predicted vs Actual Response Probabilities', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# Response Proportions by Age Group

Visualize predicted response proportions across F0 and VOT values, grouped by age.


In [ ]:
# Reload predictions with age information
predictions_with_age = pd.read_csv('../shared/output/collective_predictions.csv')

# Define age groups
def categorize_age(age):
    if age < 40:
        return 'Young (20-30s)'
    elif age < 60:
        return 'Middle (40-50s)'
    else:
        return 'Old (60+)'

predictions_with_age['age_group'] = predictions_with_age['age'].apply(categorize_age)

# Calculate actual response proportions by age group for F0
f0_actual = predictions_with_age.groupby(['f0', 'age_group', 'actual_response']).size().reset_index(name='count')
f0_actual['total'] = f0_actual.groupby(['f0', 'age_group'])['count'].transform('sum')
f0_actual['proportion'] = f0_actual['count'] / f0_actual['total']
f0_actual = f0_actual.rename(columns={'actual_response': 'response'})
f0_proportions = f0_actual[['f0', 'age_group', 'response', 'proportion']]

# Calculate actual response proportions by age group for VOT
vot_actual = predictions_with_age.groupby(['vot', 'age_group', 'actual_response']).size().reset_index(name='count')
vot_actual['total'] = vot_actual.groupby(['vot', 'age_group'])['count'].transform('sum')
vot_actual['proportion'] = vot_actual['count'] / vot_actual['total']
vot_actual = vot_actual.rename(columns={'actual_response': 'response'})
vot_proportions = vot_actual[['vot', 'age_group', 'response', 'proportion']]

print("F0 proportions sample:")
print(f0_proportions.head(10))
print("\nVOT proportions sample:")
print(vot_proportions.head(10))
print("\nAge group distribution:")
print(predictions_with_age['age_group'].value_counts().sort_index())


In [ ]:
# Define age group colors
age_colors = {
    'Young (20-30s)': '#b2e2e2',  # Light Mint
    'Middle (40-50s)': '#66c2a4',  # Medium Teal
    'Old (60+)': '#238b45'  # Deep Forest
}

# Define markers for each age group (all solid lines)
age_styles = {
    'Young (20-30s)': {'marker': 'o'},      # circle
    'Middle (40-50s)': {'marker': 's'},     # square
    'Old (60+)': {'marker': '^'}            # triangle
}

# Define response full names
response_order = ['asp', 'fortis', 'lenis']
response_titles = {
    'lenis': 'Lenis',
    'fortis': 'Fortis',
    'asp': 'Aspirated'
}

# Create 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey='row')

# First row: VOT on x-axis
for col_idx, response in enumerate(response_order):
    ax = axes[0, col_idx]
    
    # Filter data for this response
    data = vot_proportions[vot_proportions['response'] == response].copy()
    
    # Plot lines for each age group
    for age_group in ['Young (20-30s)', 'Middle (40-50s)', 'Old (60+)']:
        age_data = data[data['age_group'] == age_group].sort_values('vot')
        if len(age_data) > 0:
            ax.plot(age_data['vot'], age_data['proportion'], 
                   linewidth=2, 
                   marker=age_styles[age_group]['marker'], 
                   markersize=9,
                   alpha=0.8,
                   label=age_group, color=age_colors[age_group])
    
    # Set labels and title
    ax.set_title(response_titles[response], fontsize=14, fontweight='bold')
    ax.set_xlabel('VOT', fontsize=12, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel('Proportion', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    
    # Only show legend on first subplot
    if col_idx == 0:
        ax.legend(loc='best', fontsize=10)

# Second row: F0 on x-axis
for col_idx, response in enumerate(response_order):
    ax = axes[1, col_idx]
    
    # Filter data for this response
    data = f0_proportions[f0_proportions['response'] == response].copy()
    
    # Plot lines for each age group
    for age_group in ['Young (20-30s)', 'Middle (40-50s)', 'Old (60+)']:
        age_data = data[data['age_group'] == age_group].sort_values('f0')
        if len(age_data) > 0:
            ax.plot(age_data['f0'], age_data['proportion'], 
                   linewidth=2, 
                   marker=age_styles[age_group]['marker'], 
                   markersize=9,
                   alpha=0.8,
                   label=age_group, color=age_colors[age_group])
    
    # Set labels
    ax.set_xlabel('F0', fontsize=12, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel('Proportion', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

# Main title
fig.suptitle('Response Proportions of Each Stop Category by Age Group across VOT and F0', 
            fontsize=16, fontweight='bold', y=0.98)

# Add note at the bottom
fig.text(0.5, 0.02, 
        "Age Groups: Young (20-30s: ages 20-39), Middle (40-50s: ages 40-59), Old (60+: ages 60 and above). "
        "Proportions represent actual response frequencies for each age group across VOT and F0 values.",
        ha='center', fontsize=10, style='italic', wrap=True)

plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.show()


# Response Proportions by Gender

Visualize actual response proportions across F0 and VOT values, grouped by gender.


In [ ]:
# Reload predictions with gender information
predictions_with_gender = pd.read_csv('../shared/output/collective_predictions.csv')

# Calculate actual response proportions by gender for F0
f0_gender = predictions_with_gender.groupby(['f0', 'gender', 'actual_response']).size().reset_index(name='count')
f0_gender['total'] = f0_gender.groupby(['f0', 'gender'])['count'].transform('sum')
f0_gender['proportion'] = f0_gender['count'] / f0_gender['total']
f0_gender = f0_gender.rename(columns={'actual_response': 'response'})
f0_proportions_gender = f0_gender[['f0', 'gender', 'response', 'proportion']]

# Calculate actual response proportions by gender for VOT
vot_gender = predictions_with_gender.groupby(['vot', 'gender', 'actual_response']).size().reset_index(name='count')
vot_gender['total'] = vot_gender.groupby(['vot', 'gender'])['count'].transform('sum')
vot_gender['proportion'] = vot_gender['count'] / vot_gender['total']
vot_gender = vot_gender.rename(columns={'actual_response': 'response'})
vot_proportions_gender = vot_gender[['vot', 'gender', 'response', 'proportion']]

print("F0 proportions by gender sample:")
print(f0_proportions_gender.head(10))
print("\nVOT proportions by gender sample:")
print(vot_proportions_gender.head(10))
print("\nGender distribution:")
print(predictions_with_gender['gender'].value_counts())


In [ ]:
# Define gender colors
gender_colors = {
    'Male': "#b4b4b4",  # gray
    'Female': '#1a4e8a'  # navy
}

# Define gender markers
gender_markers = {
    'Male': 's',  # square
    'Female': 'o'  # circle
}

# Define response full names
response_order = ['asp', 'fortis', 'lenis']
response_titles = {
    'lenis': 'Lenis',
    'fortis': 'Fortis',
    'asp': 'Aspirated'
}

# Create 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharey='row')

# First row: VOT on x-axis
for col_idx, response in enumerate(response_order):
    ax = axes[0, col_idx]
    
    # Filter data for this response
    data = vot_proportions_gender[vot_proportions_gender['response'] == response].copy()
    
    # Plot lines for each gender
    for gender in ['Female', 'Male']:
        gender_data = data[data['gender'] == gender].sort_values('vot')
        if len(gender_data) > 0:
            ax.plot(gender_data['vot'], gender_data['proportion'], 
                   linewidth=2, color=gender_colors[gender], marker=gender_markers[gender], markersize=10,
                   alpha=0.8, markerfacecolor=gender_colors[gender],
                   markeredgewidth=1, markeredgecolor='#ffffff',
                   label=gender)
    
    # Set labels and title
    ax.set_title(response_titles[response], fontsize=14, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel('Response rate', fontsize=16, fontweight='bold')
    ax.set_ylim(-0.1, 1.1)
    ax.set_yticks(np.arange(0, 1.1, 0.25))
    ax.set_xticks(np.arange(1, 9, 1))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=16)
    
    # Only show legend on first subplot
    if col_idx == 0:
        legend = ax.legend(loc='best', fontsize=14, title='Gender', title_fontsize=14)
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(1.5)
    
    # Set black border for subplot
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)

# Second row: F0 on x-axis
for col_idx, response in enumerate(response_order):
    ax = axes[1, col_idx]
    
    # Filter data for this response
    data = f0_proportions_gender[f0_proportions_gender['response'] == response].copy()
    
    # Plot lines for each gender
    for gender in ['Male', 'Female']:
        gender_data = data[data['gender'] == gender].sort_values('f0')
        if len(gender_data) > 0:
            ax.plot(gender_data['f0'], gender_data['proportion'], 
                   linewidth=2, color=gender_colors[gender], marker=gender_markers[gender], markersize=10,
                   alpha=0.8, markerfacecolor=gender_colors[gender],
                   markeredgewidth=1, markeredgecolor='#ffffff',
                   label=gender)
    
    # Set labels
    ax.set_xlabel('F0', fontsize=16, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel('Response rate', fontsize=16, fontweight='bold')
    ax.set_ylim(-0.1, 1.1)
    ax.set_yticks(np.arange(0, 1.1, 0.25))
    ax.set_xticks(np.arange(1, 9, 1))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=16)
    
    # Set black border for subplot
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)

# Set xlabel for first row
for col_idx in range(3):
    axes[0, col_idx].set_xlabel('VOT', fontsize=16, fontweight='bold')

# Main title
fig.suptitle('Response rates by gender, across VOT (top) and F0 (bottom)', 
            fontsize=18, fontweight='bold', y=0.98)

# Add note at the bottom
# fig.text(0.5, 0.02, 
#         "Rates represent actual response frequencies for each gender group across VOT and F0 values.",
#         ha='center', fontsize=14, style='italic', wrap=True)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.subplots_adjust(hspace=0.25)
plt.savefig('output/response_rate_by_gender.png', dpi=300, bbox_inches='tight')
plt.show()

# Response Proportions by Place of Articulation

Visualize actual response proportions by place of articulation (PoA).


In [ ]:
# Reload predictions with poa information
predictions_with_poa = pd.read_csv('../shared/output/collective_predictions.csv')

# Calculate actual response proportions by poa
poa_response = predictions_with_poa.groupby(['actual_response', 'poa']).size().reset_index(name='count')
poa_response['total'] = poa_response.groupby('actual_response')['count'].transform('sum')
poa_response['proportion'] = poa_response['count'] / poa_response['total']
poa_response = poa_response.rename(columns={'actual_response': 'response'})

print("Response proportions by PoA:")
print(poa_response)
print("\nPoA distribution:")
print(predictions_with_poa['poa'].value_counts())


In [ ]:
# Define PoA colors (matching the image)
poa_colors = {
    'lab': "#f6b8ce",  # light pink
    'cor': "#6524bf",  # purple
    'dor': "#b4b4b4"  # gray
}

# Define PoA full names
poa_names = {
    'lab': 'Labial',
    'cor': 'Coronal',
    'dor': 'Dorsal'
}

# Define response order and names
response_order = ['asp', 'fortis', 'lenis']
response_names = {
    'lenis': 'Lenis',
    'fortis': 'Fortis',
    'asp': 'Aspirated'
}

# Prepare data for grouped bar chart
poa_types = ['lab', 'cor', 'dor']
x = np.arange(len(response_order))
width = 0.25

# Create figure
fig, ax = plt.subplots(figsize=(5, 4))

# Plot bars for each PoA and store bar containers for later modification
bar_containers = []
for i, poa in enumerate(poa_types):
    proportions = []
    for response in response_order:
        data = poa_response[(poa_response['response'] == response) & (poa_response['poa'] == poa)]
        if len(data) > 0:
            proportions.append(data['proportion'].values[0])
        else:
            proportions.append(0)
    
    bar_container = ax.bar(x + i * width, proportions, width, 
           label=poa_names[poa], color=poa_colors[poa],
           edgecolor='black', linewidth=1)
    bar_containers.append((poa, proportions, bar_container))

# Create legend BEFORE modifying bar edgecolors
legend = ax.legend(fontsize=8, title='PoA', title_fontsize=8)
legend.get_frame().set_edgecolor('black')
legend.get_frame().set_linewidth(1.5)

# Highlight the tallest bar in each response group with thicker border
for response_idx in range(len(response_order)):
    max_height = -1
    max_bar = None
    
    for poa, proportions, bar_container in bar_containers:
        if proportions[response_idx] > max_height:
            max_height = proportions[response_idx]
            max_bar = bar_container[response_idx]
    
    if max_bar:
        max_bar.set_edgecolor('black')  # Set edge color
        max_bar.set_linewidth(2)  # Thicker border for the tallest bar
        max_bar.set_zorder(10)  # Bring to front

# Customize plot
ax.set_xlabel('Category', fontsize=10, fontweight='bold')
ax.set_ylabel('Response rate', fontsize=10, fontweight='bold')
ax.set_title('Response rates by place of articulation (PoA)', fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([response_names[r] for r in response_order])
ax.tick_params(axis='both', labelsize=10)
ax.set_ylim(0, 0.55)
ax.grid(True, alpha=0.3, axis='y')
ax.set_axisbelow(True)

# Add border
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.savefig('output/response_rate_by_poa.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Prepare data
predictions_data = pd.read_csv('../shared/output/collective_predictions.csv')

# Calculate proportions by subject first, then average across subjects
def calculate_proportion_by_subject(data, group_vars, response_col='actual_response'):
    """Calculate proportion for each subject, then average across subjects with CI"""
    result_list = []
    
    for name, group in data.groupby(group_vars):
        # For each combination of group_vars, calculate proportion per subject
        subject_proportions = []
        
        for subject_id, subject_data in group.groupby('subject'):
            total = len(subject_data)
            for response in ['lenis', 'fortis', 'asp']:
                count = (subject_data[response_col] == response).sum()
                proportion = count / total if total > 0 else 0
                subject_proportions.append({
                    'subject': subject_id,
                    'response': response,
                    'proportion': proportion
                })
        
        # Convert to DataFrame for easier manipulation
        subject_df = pd.DataFrame(subject_proportions)
        
        # Calculate mean and CI across subjects for each response
        for response in ['asp', 'fortis', 'lenis']:
            response_props = subject_df[subject_df['response'] == response]['proportion']
            
            if len(response_props) > 0:
                mean_prop = response_props.mean()
                se_prop = response_props.sem()  # Standard error of the mean
                
                # 95% CI using t-distribution
                n_subjects = len(response_props)
                if n_subjects > 1:
                    from scipy import stats as scipy_stats
                    t_critical = scipy_stats.t.ppf(0.975, n_subjects - 1)
                    margin = t_critical * se_prop
                    ci_lower = max(0, mean_prop - margin)
                    ci_upper = min(1, mean_prop + margin)
                else:
                    ci_lower = ci_upper = mean_prop
            else:
                mean_prop = ci_lower = ci_upper = 0
            
            result_dict = dict(zip(group_vars, name if isinstance(name, tuple) else [name]))
            result_dict.update({
                'response': response,
                'proportion': mean_prop,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper
            })
            result_list.append(result_dict)
    
    return pd.DataFrame(result_list)

# Calculate proportions for f0-vot combinations
proportions_data = calculate_proportion_by_subject(predictions_data, ['vot', 'f0'])

# Define colors for response categories
response_colors = {
    'lenis': '#1f77b4',    # blue
    'fortis': '#ff7f0e',   # orange
    'asp': '#2ca02c'       # green
}

response_labels = {
    'lenis': 'Lenis',
    'fortis': 'Fortis',
    'asp': 'Aspirated'
}

# Create 2x4 subplots
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex='row', sharey=True)

# First row: VOT on x-axis, different f0 values (1, 3, 5, 7)
f0_values = [1, 3, 5, 7]
for col_idx, f0_val in enumerate(f0_values):
    ax = axes[0, col_idx]
    
    # Filter data for this f0 value
    data_subset = proportions_data[proportions_data['f0'] == f0_val]
    
    # Plot each response category
    for response in ['asp', 'fortis', 'lenis']:
        response_data = data_subset[data_subset['response'] == response].sort_values('vot')
        
        if len(response_data) > 0:
            # Plot line
            ax.plot(response_data['vot'], response_data['proportion'],
                   color=response_colors[response], linewidth=2.5,
                   marker='o', markersize=8, alpha=0.8, label=response_labels[response],
                   markeredgewidth=1, markeredgecolor='white')
            
            # Plot confidence interval
            ax.fill_between(response_data['vot'], 
                           response_data['ci_lower'],
                           response_data['ci_upper'],
                           color=response_colors[response], alpha=0.2)
    
    # Customize subplot
    ax.set_title(f'F0 = {f0_val}', fontsize=16, fontweight='bold')
    ax.set_yticks(np.arange(0, 1.1, 0.25))
    ax.set_xticks(np.arange(1, 8.1, 1))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=16)
    
    if col_idx == 0:
        ax.set_ylabel('Response rate', fontsize=16, fontweight='bold')
    
    # Add legend to 4th subplot
    if col_idx == 3:
        legend = ax.legend(loc='right', fontsize=14, framealpha=0.9, title='Category', title_fontsize=14)
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(1.5)
    
    # Set black border
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)

# Second row: f0 on x-axis, different vot values (1, 3, 5, 7)
vot_values = [1, 3, 5, 7]
for col_idx, vot_val in enumerate(vot_values):
    ax = axes[1, col_idx]
    
    # Filter data for this vot value
    data_subset = proportions_data[proportions_data['vot'] == vot_val]
    
    # Plot each response category
    for response in ['asp', 'fortis', 'lenis']:
        response_data = data_subset[data_subset['response'] == response].sort_values('f0')
        
        if len(response_data) > 0:
            # Plot line
            ax.plot(response_data['f0'], response_data['proportion'],
                   color=response_colors[response], linewidth=2.5,
                   marker='o', markersize=8, alpha=0.8, label=response_labels[response],
                   markeredgewidth=1, markeredgecolor='white')
            
            # Plot confidence interval
            ax.fill_between(response_data['f0'], 
                           response_data['ci_lower'],
                           response_data['ci_upper'],
                           color=response_colors[response], alpha=0.2)
    
    # Customize subplot
    ax.set_title(f'VOT = {vot_val}', fontsize=16, fontweight='bold')
    ax.set_xlabel('F0', fontsize=16, fontweight='bold')
    ax.set_yticks(np.arange(0, 1.1, 0.25))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=16)

    # Set black border
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    
    if col_idx == 0:
        ax.set_ylabel('Response rate', fontsize=16, fontweight='bold')

# Set x-label for first row
for col_idx in range(4):
    axes[0, col_idx].set_xlabel('VOT', fontsize=16, fontweight='bold')

# Main title
fig.suptitle('Response rates across VOT and F0, with the other dimension fixed at steps 1, 3, 5, and 7',
            fontsize=18, fontweight='bold', y=0.995)

plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Adjust top margin and vertical spacing
plt.savefig('output/response_rate_vot_f0.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create schematic grid plot
# VOT: 10 to 115 in steps of 15
# f0: 135 to 205 in steps of 10

vot_values = np.arange(10, 116, 15)  # [10, 25, 40, 55, 70, 85, 100, 115]
f0_values = np.arange(135, 206, 10)  # [135, 145, 155, 165, 175, 185, 195, 205]

# Create figure with square aspect ratio
fig, ax = plt.subplots(figsize=(8, 8))

# Create grid of points
for f0 in f0_values:
    for vot in vot_values:
        ax.plot(vot, f0, 'o', color='black', markersize=8)

# Set labels and formatting
ax.set_xlabel('VOT (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('f0 (Hz)', fontsize=12, fontweight='bold')
ax.set_title('Schematic Grid: VOT × f0', fontsize=14, fontweight='bold')

# Set axis limits with some padding
ax.set_xlim(0, 125)
ax.set_ylim(125, 215)

# Set ticks to exact VOT and f0 values
ax.set_xticks(vot_values)
ax.set_yticks(f0_values)

# Remove grid
ax.grid(False)

# Add border
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.show()

print(f"VOT values: {vot_values}")
print(f"f0 values: {f0_values}")
print(f"Total points: {len(vot_values)} × {len(f0_values)} = {len(vot_values) * len(f0_values)}")